# Lab 02 — Análisis (Equipo 2, Nivel C)

Este notebook **no contiene lógica**: lee los resultados que generó `python main.py` en `docs/resultados/` y muestra las figuras que produce `src/plots.py`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from src import config

RES = RAIZ / "docs" / "resultados"
FIG = RAIZ / "docs" / "figures"
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


def csv(nombre):
    return pd.read_csv(RES / nombre)


def leer_json(nombre):
    ruta = RES / nombre
    return json.loads(ruta.read_text(encoding="utf-8")) if ruta.exists() else None


def ver(*nombres):
    for nombre in nombres:
        ruta = FIG / nombre
        display(Image(filename=str(ruta))) if ruta.exists() else print(f"(figura pendiente: {nombre})")

## 1. Datos

In [ ]:
print("Activos por integrante:", config.ASIGNACION)
display(csv("auditoria_datos.csv"))
leer_json("splits.json")

## 2. Estrategia (3 indicadores, regla 2 de 3)

In [ ]:
ver("13_correlacion_senales.png", "09_dos_de_tres_vs_un_indicador.png")
csv("un_indicador.csv").query("nivel == 'portafolio'")

## 3. Régimen

In [ ]:
validacion = leer_json("regimenes_validacion.json")
print("Features:", validacion["features_usadas"], "| silhouette:", round(validacion["silhouette_modelo_congelado"], 3))
display(csv("regimen_persistencia_wf_oos.csv"))
ver("06a_regimenes_linea_tiempo.png", "06b_features_por_regimen.png",
    "06c_valor_con_regimenes.png", "06d_transiciones.png")
csv("metricas_por_regimen_wf_oos.csv")

## 4. Portafolio (Risk Parity)

In [ ]:
display(csv("rp_vs_naive_vs_ew.csv")[["nombre", "cagr", "mdd", "calmar", "rc_max", "rc_tsla_nvda"]])
display(csv("estimadores_sigma.csv"))
ver("07a_contribuciones_riesgo.png", "07b_senales_s_wf_oos.png", "07c_correlacion_por_regimen.png",
    "07d_barrido_rebalanceo.png", "08_pesos_estimadores_sigma.png")

## 5. Optimización y walk-forward

In [ ]:
resumen = leer_json("optimizacion_resumen.json")
print(f"Trials totales en TRAIN: {resumen['trials_totales_train']:,}")
print(f"Tiempo de optimización: {resumen['segundos_reloj_optimizacion'] / 3600:.2f} h de reloj")
variantes = csv("comparacion_variantes.csv")
display(variantes[["nombre", "cagr", "mdd", "calmar", "wfe_rendimiento", "wfe_calmar"]])
for _, fila in variantes.iterrows():
    print(f"WFE {fila['nombre']}: rendimiento {fila['wfe_rendimiento']:.2f} | Calmar {fila['wfe_calmar']:.2f}")
ver("01_valor_portafolio.png", "02_drawdown.png", "03_rendimientos_wf_oos.png", "12_rolling_vs_anchored.png",
    "14a_optuna_historia.png", "14b_optuna_importancia.png", "14c_optuna_slices.png",
    "14d_optuna_superficie_3d.png")
csv("metricas_wf_oos.csv")

## 6. Robustez y costos

In [ ]:
print(leer_json("costos_resumen.json"))
display(csv("costos_realistas_train.csv"))
ver("04_sensibilidad.png", "05_curva_costos.png", "11_costos_vs_bruto_wf_oos.png",
    "10_portafolio_vs_individuales_wf_oos.png")

## 7. TEST (se toca una sola vez)

In [ ]:
lock = leer_json("test_lock.json")
if lock is None:
    print("Commit del test: pendiente")
else:
    print("Commit del test:", lock["commit"], "| fecha:", lock["fecha"])
    display(csv("metricas_test.csv"))
    display(csv("test_secundario_wf.csv"))
    ver("03_rendimientos_test.png")

## 8. Respuestas a las preguntas

Las respuestas completas, con cifras leídas de `docs/resultados/`, están en la sección 13 de `docs/borrador_reporte.md` (generado por `src/report.py`).

In [ ]:
texto = (RAIZ / "docs" / "borrador_reporte.md").read_text(encoding="utf-8")
inicio = texto.find("## 13.")
display(Markdown(texto[inicio:texto.find("## 14.")]))